# [LAB 12] 3-1. 회귀분석 결과보고의 이해

## #01. 준비작업

### 1. 패키지 참조

In [1]:
# 라이브러리 기본 참조
from jussam import load_data
from helpers import *
from pandas import DataFrame
from IPython.display import display, Markdown
from statsmodels.stats.stattools import durbin_watson

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 데이터 가져오기

In [2]:
origin = load_data('cars')
origin.head()

📚 자동차의 속도(speed)에 따른 제동거리(dist) 조사 데이터 (출처: R 기본 데이터)


,speed,dist
0,4,2
1,4,10
2,7,4
3,7,22
4,8,16


### 3. 단순선형회귀

In [3]:
fit = my_ols.fit_model(origin, 'dist', summary=True)

                            OLS Regression Results                            
Dep. Variable:                   dist   R-squared:                       0.651
Model:                            OLS   Adj. R-squared:                  0.644
Method:                 Least Squares   F-statistic:                     89.57
Date:                Thu, 23 Jul 2026   Prob (F-statistic):           1.49e-12
Time:                        07:43:49   Log-Likelihood:                -206.58
No. Observations:                  50   AIC:                             417.2
Df Residuals:                      48   BIC:                             421.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -17.5791      6.758     -2.601      0.0

## #02. 모형 적합도 보고

### 1. 독립변수의 이름 확인

In [4]:
# 분석에 사용된 전체 독립변수 이름
# 상수항(const)이 포함되어 있다.
print(fit.model.exog_names)

# 상수항을 제외한 나머지 독립변수 이름
# -> 상수항 이름만 제외한 독립변수 이름만 조회하는 전용 속성은 없으므로 exog_names에서 걸러낸다.
xnames = [name for name in fit.model.exog_names if name != 'const']
print(xnames)

['const', 'speed']
['speed']


### 2. 모형 적합도 문장 템플릿

In [5]:
# 괄호로 묶으면 여러 문자열을 하나의 문자열로 합칠 수 있다.
template = (
    "**Note. n = {n}. "
    "F({df_model}, {df_resid}) = {f_value}, "
    "p {alpha}, "
    "R² = {r_squared}, "
    "Adj.R² = {adj_r_squared}, "
    "Durbin-Watson = {durbin_watson}**\n\n"
    "{Y}를 종속변수로, {X}(을)를 독립변수로한 {type}회귀분석 결과, 모형은 통계적으로 {result}.\n\n"
    "> F({df_model}, {df_resid}) = {f_statistic}, p {alpha}, R² = {r_squared}.\n\n"
    "즉, {X}는 {Y}의 약 {r_squared_percent}%를 설명하는 것으로 나타났다."
)

# markdown 객체로 변환하여 출력
display(Markdown(template))

**Note. n = {n}. F({df_model}, {df_resid}) = {f_value}, p {alpha}, R² = {r_squared}, Adj.R² = {adj_r_squared}, Durbin-Watson = {durbin_watson}**

{Y}를 종속변수로, {X}(을)를 독립변수로한 {type}회귀분석 결과, 모형은 통계적으로 {result}.

> F({df_model}, {df_resid}) = {f_statistic}, p {alpha}, R² = {r_squared}.

즉, {X}는 {Y}의 약 {r_squared_percent}%를 설명하는 것으로 나타났다.

### 3. 유의수준 판별

In [6]:
p = fit.f_pvalue

if p < 0.001:
    alpha = "< 0.001"
elif p < 0.01:
    alpha = "< 0.01"
elif p < 0.05:
    alpha = "< 0.05"
else:
    alpha = "≥ 0.05"

alpha

'< 0.001'

### 4. 모형 적합도 보고 문장 생성

In [7]:
report = template.format(
    n=int(fit.nobs),                                    # 관측치 수(n)
    df_model=int(fit.df_model),                         # 모형 자유도
    df_resid=int(fit.df_resid),                         # 잔차 자유도
    f_value=round(fit.fvalue, 2),                       # F 통계량
    p_value=fit.f_pvalue,                               # F 유의확률(완전 정밀도)
    r_squared=round(fit.rsquared, 3),                   # 결정계수 R²
    adj_r_squared=round(fit.rsquared_adj, 3),           # 수정된 결정계수
    durbin_watson=round(durbin_watson(fit.wresid), 3),  # 더빈-왓슨 통계량
    Y=fit.model.endog_names,                            # 종속변수 이름
    X=", ".join(xnames),                                # 독립변수 이름 나열
    type="단순선형" if len(xnames) == 1 else "다중선형",   # 회귀분석 유형
    f_statistic=round(fit.fvalue, 2),                   # F 통계량(문장 두 번째 등장)
    alpha=alpha,                                        # 유의수준 구간 표기
    r_squared_percent=round(fit.rsquared * 100, 2),     # 설명력(%) = R² × 100
    result="유의하였다" if fit.f_pvalue < 0.05 else "유의하지 않았다"  # 모형 유의성 판정
)

display(Markdown(report))

**Note. n = 50. F(1, 48) = 89.57, p < 0.001, R² = 0.651, Adj.R² = 0.644, Durbin-Watson = 1.676**

dist를 종속변수로, speed(을)를 독립변수로한 단순선형회귀분석 결과, 모형은 통계적으로 유의하였다.

> F(1, 48) = 89.57, p < 0.001, R² = 0.651.

즉, speed는 dist의 약 65.11%를 설명하는 것으로 나타났다.

## #03. 독립변수 보고

### 1. 독립변수에 대한 VIF값 계산

- my_stats 모듈의 compute_vif 함수를 사용하여 VIF는 독립변수 전체를 대상으로 한 번에 계산한다.
  - 독립변수가 하나뿐인 단순선형회귀에서는 VIF가 1.0이 된다.

In [8]:
vif = my_stats.compute_vif(origin, columns=xnames)
vif

,VIF
speed,1.000


### 2. 독립변수 보고 표 생성

In [9]:
yname = fit.model.endog_names   # 종속변수 이름
variables = []   # 분할된 내용을 저장할 빈 리스트

for x in xnames:
    # 미리 계산해 둔 VIF 표에서 해당 독립변수의 값을 조회
    vif_value = vif.loc[x, "VIF"]

    # 계수 관련 수치는 요약 문자열(반올림된 표시값)을 파싱하는 대신
    # fit 객체에서 완전한 정밀도의 실수값으로 직접 가져온다.
    variables.append({
        "종속변수": yname,                       # 종속변수 이름
        "독립변수": x,                           # 독립변수 이름
        "B": fit.params[x],                     # 비표준화 회귀계수(B)
        "표준오차": fit.bse[x],                  # 계수 표준오차
        "β": (float(fit.params[x]) * (origin[x].std(ddof=1) / 
              origin[yname].std(ddof=1))),      # 표준화 회귀계수(β)
        "t": fit.tvalues[x],                    # t-통계량
        "유의확률": fit.pvalues[x],              # 계수 유의확률
        "공차": 1 / vif_value,                  # 공차
        "VIF": vif_value,                      # 분산팽창계수
    })

variable_df = DataFrame(variables)
variable_df

,종속변수,독립변수,B,표준오차,β,t,유의확률,공차,VIF
0,dist,speed,3.932,0.416,0.807,9.464,0.000,1.000,1.000


### 3. 독립변수 보고 서술 문장 템플릿

In [10]:
template = ("- **{x}**의 회귀계수는 **B = {B}**으로 나타났으며, "
            "이는 **{yname}**에 {sig_word} 요인임을 의미한다. "
            "(**t({df_resid}) = {t}**, **{p_text}**) "
            "즉, {x}가 **1 증가**할 때 {yname}는 평균적으로 "
            "**{abs} {direction}**하는 것으로 해석된다.")

### 4. 독립변수 보고 서술

In [11]:
for x in xnames:
    if fit.pvalues[x] < 0.001:      alpha = "< 0.001"
    elif fit.pvalues[x] < 0.01:     alpha = "< 0.01"
    elif fit.pvalues[x] < 0.05:     alpha = "< 0.05"
    else:                           alpha = "≥ 0.05"

    display(Markdown(template.format(
        x=x,
        B=round(fit.params[x], 2),
        yname=yname,
        sig_word="유의한" if fit.pvalues[x] < 0.05 else "유의하지 않은",
        df_resid=int(fit.df_resid),
        t=round(fit.tvalues[x], 2),
        p_text=alpha,
        abs=round(abs(fit.params[x]), 2),
        direction="증가" if fit.params[x] > 0 else "감소"
    )))

- **speed**의 회귀계수는 **B = 3.93**으로 나타났으며, 이는 **dist**에 유의한 요인임을 의미한다. (**t(48) = 9.46**, **< 0.001**) 즉, speed가 **1 증가**할 때 dist는 평균적으로 **3.93 증가**하는 것으로 해석된다.